In [ ]:
# ======================================================
# Notebook: 2D contamination detection using CNN
# Bayesian-style sampling via MC Dropout uncertainty
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")     # (N,)
y_bin = (y > 0).astype(np.float32)

# Reshape for CNN: (batch, channel=1, height=2, width=1)
X_cnn = X.reshape(-1, 1, 2, 1)

X_t = torch.tensor(X_cnn, dtype=torch.float32)
y_t = torch.tensor(y_bin.reshape(-1,1), dtype=torch.float32)

# Simple CNN
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(2,1)),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.fc = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(500):
    optimizer.zero_grad()
    preds = model(X_t)
    loss = criterion(preds, y_t)
    loss.backward()
    optimizer.step()

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T
X_grid_cnn = X_grid.reshape(-1,1,2,1)

X_grid_t = torch.tensor(X_grid_cnn, dtype=torch.float32)

# MC Dropout for uncertainty
model.train()
mc_preds = []

with torch.no_grad():
    for _ in range(30):
        mc_preds.append(model(X_grid_t).numpy())

mc_preds = np.stack(mc_preds)
mean_prob = mc_preds.mean(axis=0).flatten()
uncertainty = mc_preds.std(axis=0).flatten()

# Select next (10,2) inputs
top_idx = np.argsort(uncertainty)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, mean_prob.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()